Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [1]:
# #@title Environment Setup

from pathlib import Path

OPTIONS = {}

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
USE_COMFYUI_MANAGER = True  #@param {type:"boolean"}
INSTALL_CUSTOM_NODES_DEPENDENCIES = True  #@param {type:"boolean"}
OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI
OPTIONS['USE_COMFYUI_MANAGER'] = USE_COMFYUI_MANAGER
OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES'] = INSTALL_CUSTOM_NODES_DEPENDENCIES

current_dir = !pwd
WORKSPACE = f"{current_dir[0]}/ComfyUI"

if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /

    from google.colab import drive
    drive.mount('/content/drive')

    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive

![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-

  # Correction of the issue of permissions being deleted on Google Drive.
  ![ -f ".ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/nightly/windows_base_files/run_nvidia_gpu.bat" ] && chmod 755 .ci/nightly/windows_base_files/run_nvidia_gpu.bat
  ![ -f ".ci/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/update_windows/update.py" ] && chmod 755 .ci/update_windows/update.py
  ![ -f ".ci/update_windows/update_comfyui.bat" ] && chmod 755 .ci/update_windows/update_comfyui.bat
  ![ -f ".ci/update_windows/README_VERY_IMPORTANT.txt" ] && chmod 755 .ci/update_windows/README_VERY_IMPORTANT.txt
  ![ -f ".ci/update_windows/run_cpu.bat" ] && chmod 755 .ci/update_windows/run_cpu.bat
  ![ -f ".ci/update_windows/run_nvidia_gpu.bat" ] && chmod 755 .ci/update_windows/run_nvidia_gpu.bat

  !git pull

!echo -= Install dependencies =-
!pip3 install accelerate
!pip3 install einops transformers>=4.28.1 safetensors>=0.4.2 aiohttp pyyaml Pillow scipy tqdm psutil tokenizers>=0.13.3
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip3 install torchsde
!pip3 install kornia>=0.7.1 spandrel soundfile sentencepiece

if OPTIONS['USE_COMFYUI_MANAGER']:
  %cd custom_nodes

  # Correction of the issue of permissions being deleted on Google Drive.
  ![ -f "ComfyUI-Manager/check.sh" ] && chmod 755 ComfyUI-Manager/check.sh
  ![ -f "ComfyUI-Manager/scan.sh" ] && chmod 755 ComfyUI-Manager/scan.sh
  ![ -f "ComfyUI-Manager/node_db/dev/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/dev/scan.sh
  ![ -f "ComfyUI-Manager/node_db/tutorial/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/tutorial/scan.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-win.bat" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-win.bat

  ![ ! -d ComfyUI-Manager ] && echo -= Initial setup ComfyUI-Manager =- && git clone https://github.com/ltdrdata/ComfyUI-Manager
  %cd ComfyUI-Manager
  !git pull

%cd $WORKSPACE

if OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES']:
  !echo -= Install custom nodes dependencies =-
  !pip install GitPython
  !python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies


Mounting Google Drive...
/
Mounted at /content/drive
/content/drive/MyDrive
/content/drive/MyDrive/ComfyUI
-= Updating ComfyUI =-
Already up to date.
-= Install dependencies =-
Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 5.1 MB/s eta 0:00:00
/content/drive/MyDrive/ComfyUI/custom_nodes
/content/drive/MyDrive/ComfyUI/custom_nodes/ComfyUI-Manager
There is no tracking information for the current branch.
Please specify which branch you want to merge with.
See git-pull(1) for details.

    git pull <remote> <branch>

If you wish to set tracking information for this branch you can do so with:

    git branch --set-upstream-to=origin/<branch> master

/content/drive/MyDrive/ComfyUI
-= Install custom nodes dependencies =-
python3: can't open file '/content/drive/MyDrive/ComfyUI/custom_nodes/ComfyUI-Manager/cm-cli.py': [Errno 2] No such file or directory


Download some models/checkpoints/vae or custom comfyui nodes (uncomment the commands for the ones you want)

In [8]:
# ============================================
# SDXL MODELS (ОСНОВА)
# ============================================

# SDXL Base (основная модель)
!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors -P ./models/checkpoints/

# SDXL VAE (ОБЯЗАТЕЛЬНО для SDXL!)
!wget -c https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors -P ./models/vae/

# ============================================
# UPSCALE МОДЕЛИ
# ============================================

# 4x NMKD UltraYandere (лучший для фотореализма)
!wget -c https://huggingface.co/gemasai/4x_NMKD-UltraYandere_300k/resolve/main/4x_NMKD-UltraYandere_300k.pth -P ./models/upscale_models/

# 4x UltraSharp (альтернатива, более резкий)
!wget -c https://huggingface.co/Kim2091/UltraSharp/resolve/main/4x-UltraSharp.pth -P ./models/upscale_models/

# ============================================
# FACE RESTORATION МОДЕЛИ
# ============================================

# GFPGAN (восстановление лица)
!wget -c https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth -P ./models/facerestore_models/

# CodeFormer (альтернатива GFPGAN)
!wget -c https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth -P ./models/facerestore_models/

# ============================================
# INSIGHTFACE (для ReActor)
# ============================================

!mkdir -p ./models/insightface
!wget -c https://huggingface.co/MonsterMMORPG/tools/resolve/main/inswapper_128.onnx -P ./models/insightface/

# ============================================
# YOLO МОДЕЛИ (для FaceDetailer/HandDetailer)
# ============================================

!mkdir -p ./models/ultralytics

# Face detection
!wget -c https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8n.pt -P ./models/ultralytics/

# Hand detection
!wget -c https://huggingface.co/Bingsu/adetailer/resolve/main/hand_yolov8n.pt -P ./models/ultralytics/

# Person detection (опционально)
!wget -c https://huggingface.co/Bingsu/adetailer/resolve/main/yolov8n.pt -P ./models/ultralytics/

# ============================================
# CONTROLNET SDXL (опционально, для будущих задач)
# ============================================

# ControlNet Canny для SDXL
!wget -c https://huggingface.co/diffusers/controlnet-canny-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors -O ./models/controlnet/controlnet-canny-sdxl.safetensors

# ControlNet Depth для SDXL
!wget -c https://huggingface.co/diffusers/controlnet-depth-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors -O ./models/controlnet/controlnet-depth-sdxl.safetensors

print("✅ Все модели скачаны!")

--2026-06-28 14:19:05--  https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors
Resolving huggingface.co (huggingface.co)... 18.164.174.23, 18.164.174.17, 18.164.174.55, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/64bfcd5ff462a99a04fd1ec8/3d6f740fa52572e1071b8ecb7c5f8a8e2cbef596a51121102877bd9900078891?user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27sd_xl_base_1.0.safetensors%3B+filename%3D%22sd_xl_base_1.0.safetensors%22%3B&X-Xet-Cas-Uid=public&Expires=1782659945&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRiZmNkNWZmNDYyYTk5YTA0ZmQxZWM4LzNkNmY3NDBmYTUyNTcyZTEwNzFiOGVjYjdjNWY4YThlMmNiZWY1OTZhNTExMjExMDI4NzdiZDk5MDAwNzg4OTFcXD91c2VyX2lkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPWlubGluZSUzQitmaWxlbmFtZSUyQSUzRFVURi04

In [10]:
# Создаём папки
!mkdir -p ./models/insightface
!mkdir -p ./models/controlnet
!mkdir -p ./models/ultralytics

# inswapper_128.onnx (правильная ссылка)
!wget -c https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx -P ./models/insightface/

# 4x_NMKD-UltraYandere (из OpenModelDB, без авторизации)
!wget -c https://huggingface.co/subby2006/NMKD-UltraYandere/resolve/7aa118f5328a3a01deb095e6b11cda9d0ac7dd7f/4x_NMKD-UltraYandere_300k.pth || echo "NMKD не скачался, используем UltraSharp"

# Если NMKD не скачался, можно использовать RealESRGAN как альтернативу
!wget -c https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P ./models/upscale_models/

# ControlNet SDXL (правильные ссылки)
!wget -c https://huggingface.co/diffusers/controlnet-canny-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors -O ./models/controlnet/controlnet-canny-sdxl.safetensors

!wget -c https://huggingface.co/diffusers/controlnet-depth-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors -O ./models/controlnet/controlnet-depth-sdxl.safetensors

print("✅ Исправленные модели скачаны!")

--2026-06-28 14:32:40--  https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx
Resolving huggingface.co (huggingface.co)... 18.164.174.23, 18.164.174.118, 18.164.174.55, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/64a4ee92f489f7ced55b6c7e/34890f0ff08211a8af3cfd4a6f216f0f64c7ed044a8c136410134695142e30c8?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27inswapper_128.onnx%3B+filename%3D%22inswapper_128.onnx%22%3B&X-Xet-Cas-Uid=public&user_id=public&Expires=1782660760&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRhNGVlOTJmNDg5ZjdjZWQ1NWI2YzdlLzM0ODkwZjBmZjA4MjExYThhZjNjZmQ0YTZmMjE2ZjBmNjRjN2VkMDQ0YThjMTM2NDEwMTM0Njk1MTQyZTMwYzhcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPWlubGluZSUzQitmaWxlbmFtZSUyQSUzRFVURi04JTI3JTI3aW5zd2FwcGVyXzEyOC5vbm54JTNCK2ZpbGVuYW1lJTNEJTIy

### Run ComfyUI with cloudflared (Recommended Way)




In [6]:
!wget -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI: ", l[l.find("http"):], end='')

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

--2026-06-28 18:46:50--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.6.1/cloudflared-linux-amd64.deb [following]
--2026-06-28 18:46:50--  https://github.com/cloudflare/cloudflared/releases/download/2026.6.1/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/7e5346eb-2435-4634-bcbb-e4e22e283cd7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-28T19%3A35%3A48Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

In [4]:
!pip install -r /content/drive/MyDrive/ComfyUI/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 176.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 275.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 258.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.0/64.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.9/383.9 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 238.7 MB/s eta 0:00:00


In [25]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 24.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 86.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 73.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 48.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 97.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1

In [2]:

!wget https://huggingface.co/datasets/tyDiffusion/Diffusion/resolve/2e577588360c68e422c687922feb5e018aab7e2f/realvisXL-4.0.safetensors -O ./models/checkpoints/

./models/checkpoints/: Is a directory


### Run ComfyUI with localtunnel




In [5]:
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

⠙
changed 22 packages in 236ms
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[WARNING] WARNING: You need pytorch with cu130 or higher to use optimized CUDA operations.
[INFO] Found comfy_kitchen backend eager: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope_split_half', 'apply_rope_split_half1', 'dequantize_int8_simple', 'dequantize_mxfp8', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'gemv_awq_w4a16', 'int8_linear', 'quantize_and_rotate_rowwise', 'quantize_int8_rowwise', 'quantize_int8_tensorwise', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'quantize_svdquant_w4a

### Run ComfyUI with colab iframe (use only in case the previous way with localtunnel doesn't work)

You should see the ui appear in an iframe. If you get a 403 error, it's your firefox settings or an extension that's messing things up.

If you want to open it in another window use the link.

Note that some UI features like live image previews won't work because the colab iframe blocks websockets.

In [12]:
import threading
import time
import socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  from google.colab import output
  output.serve_kernel_port_as_iframe(port, height=1024)
  print("to open it in a window you can open this link here:")
  output.serve_kernel_port_as_window(port)

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[WARNING] WARNING: You need pytorch with cu130 or higher to use optimized CUDA operations.
[INFO] Found comfy_kitchen backend cuda: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope_split_half', 'apply_rope_split_half1', 'dequantize_int8_simple', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'gemv_awq_w4a16', 'int8_linear', 'quantize_and_rotate_rowwise', 'quantize_int8_rowwise', 'quantize_int8_tensorwise', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'quantize_svdquant_w4a4', 'scaled_mm_nvfp4', 'scaled_mm_svdquant_w4a4', 'stochastic_rounding_fp8']}
[INFO] Found comfy_kitchen backend triton: {

<IPython.core.display.Javascript object>

to open it in a window you can open this link here:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

[INFO] 
Stopped server


In [1]:
#@title Environment Setup

In [28]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [19]:
import os
from pathlib import Path

# Путь к ComfyUI (зависит от настроек ноутбука)
COMFYUI_PATH = "/content/drive/MyDrive/ComfyUI"

# Если ComfyUI установлен локально (не в Drive), раскомментируй:
# COMFYUI_PATH = "/content/ComfyUI"

# Проверка существования папки
if not os.path.exists(COMFYUI_PATH):
    print(f"❌ ComfyUI не найден по пути: {COMFYUI_PATH}")
    print(" Проверь параметр USE_GOOGLE_DRIVE в ячейке Environment Setup")
else:
    print(f"✅ ComfyUI найден: {COMFYUI_PATH}\n")

# Словарь с моделями для проверки
models_to_check = {
    "📦 CHECKPOINTS": {
        "sd_xl_base_1.0.safetensors": "models/checkpoints/",
        "v1-5-pruned-emaonly.ckpt": "models/checkpoints/",  # SD1.5 (если нужен)
    },

    "🎨 VAE": {
        "sdxl_vae.safetensors": "models/vae/",
        "vae-ft-mse-840000-ema-pruned.safetensors": "models/vae/",  # SD1.5 VAE
    },

    "🔍 UPSCALE MODELS": {
        "4x-UltraSharp.pth": "models/upscale_models/",
        "RealESRGAN_x4plus.pth": "models/upscale_models/",
        "4x_NMKD-UltraYandere_300k.pth": "models/upscale_models/",
    },

    "👤 FACE RESTORATION": {
        "GFPGANv1.4.pth": "models/facerestore_models/",
        "codeformer.pth": "models/facerestore_models/",
    },

    "🔄 INSIGHTFACE (ReActor)": {
        "inswapper_128.onnx": "models/insightface/",
    },

    " ULTRALYTICS (Detailer)": {
        "face_yolov8n.pt": "models/ultralytics/",
        "hand_yolov8n.pt": "models/ultralytics/",
        "yolov8n.pt": "models/ultralytics/",
    },

    "🕸️ CONTROLNET SDXL": {
        "controlnet-canny-sdxl.safetensors": "models/controlnet/",
        "controlnet-depth-sdxl.safetensors": "models/controlnet/",
    },

    "🌟 LORA (Хана)": {
        "hana-000001.safetensors": "models/loras/",
        "hana-000005.safetensors": "models/loras/",
        "hana-000010.safetensors": "models/loras/",
    },
}

# Проверка каждой модели
total_models = 0
installed_models = 0
missing_models = []

for category, models in models_to_check.items():
    print(f"\n{'─' * 50}")
    print(f"  {category}")
    print(f"{'─' * 50}")

    for model_name, folder in models.items():
        total_models += 1
        full_path = os.path.join(COMFYUI_PATH, folder, model_name)

        if os.path.exists(full_path):
            # Получаем размер файла в MB
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
            installed_models += 1
            print(f"  ✅ {model_name:<45} {size_mb:.1f} MB")
        else:
            missing_models.append((category, model_name, folder))
            print(f"  ❌ {model_name:<45} ОТСУТСТВУЕТ")

# Итоговая сводка
print(f"\n{'═' * 50}")
print(f"  📊 ИТОГОВАЯ СТАТИСТИКА")
print(f"{'═' * 50}")
print(f"  ✅ Установлено: {installed_models}/{total_models}")
print(f"  ❌ Отсутствует: {len(missing_models)}")
print(f"  📈 Процент: {(installed_models/total_models*100):.1f}%")

if missing_models:
    print(f"\n{'⚠️' * 20}")
    print("  ОТСУТСТВУЮЩИЕ МОДЕЛИ:")
    print(f"{'⚠️' * 20}")
    for category, model_name, folder in missing_models:
        print(f"  • [{category}] {model_name}")
        print(f"    Папка: {folder}")
else:
    print(f"\n  🎉 ВСЕ МОДЕЛИ УСТАНОВЛЕНЫ! Можно запускать ComfyUI!")

# Дополнительная проверка: список всех LoRA
print(f"\n{'─' * 50}")
print("  📋 СПИСОК ВСЕХ LORA ФАЙЛОВ:")
print(f"{'─' * 50}")
lora_folder = os.path.join(COMFYUI_PATH, "models/loras/")
if os.path.exists(lora_folder):
    lora_files = [f for f in os.listdir(lora_folder) if f.endswith('.safetensors')]
    if lora_files:
        for lora in sorted(lora_files):
            size_mb = os.path.getsize(os.path.join(lora_folder, lora)) / (1024 * 1024)
            print(f"  🌟 {lora:<45} {size_mb:.1f} MB")
    else:
        print("  ️  Папка loras пустая!")
else:
    print("  ❌ Папка models/loras/ не существует")

✅ ComfyUI найден: /content/drive/MyDrive/ComfyUI


──────────────────────────────────────────────────
  📦 CHECKPOINTS
──────────────────────────────────────────────────
  ✅ sd_xl_base_1.0.safetensors                    6616.7 MB
  ✅ v1-5-pruned-emaonly.ckpt                      4067.8 MB

──────────────────────────────────────────────────
  🎨 VAE
──────────────────────────────────────────────────
  ✅ sdxl_vae.safetensors                          319.1 MB
  ✅ vae-ft-mse-840000-ema-pruned.safetensors      319.1 MB

──────────────────────────────────────────────────
  🔍 UPSCALE MODELS
──────────────────────────────────────────────────
  ✅ 4x-UltraSharp.pth                             63.9 MB
  ✅ RealESRGAN_x4plus.pth                         63.9 MB
  ✅ 4x_NMKD-UltraYandere_300k.pth                 0.0 MB

──────────────────────────────────────────────────
  👤 FACE RESTORATION
──────────────────────────────────────────────────
  ✅ GFPGANv1.4.pth                                332.5 MB
  ✅ 

In [18]:
!wget https://huggingface.co/Ultralytics/YOLOv8/resolve/main/yolov8n.pt  -P ./models/ultralytics/
!wget https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/
!wget https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt -P ./models/checkpoints/

--2026-06-28 14:45:08--  https://huggingface.co/Ultralytics/YOLOv8/resolve/main/yolov8n.pt
Resolving huggingface.co (huggingface.co)... 18.164.174.23, 18.164.174.55, 18.164.174.118, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/65ba5b9739bd6235f7750cb9/a0bee066069f7bc71c3ddf20a6f9fbc9a511c322c87650abe42989123b471a27?user_id=public&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27yolov8n.pt%3B+filename%3D%22yolov8n.pt%22%3B&Expires=1782661508&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjViYTViOTczOWJkNjIzNWY3NzUwY2I5L2EwYmVlMDY2MDY5ZjdiYzcxYzNkZGYyMGE2ZjlmYmM5YTUxMWMzMjJjODc2NTBhYmU0Mjk4OTEyM2I0NzFhMjdcXD91c2VyX2lkPXB1YmxpYyZYLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPWlubGluZSUzQitmaWxlbmFtZSUyQSUzRFVURi04JTI3JTI3eW9sb3Y4bi5wdCUzQitmaWxlbmFtZSUzR